In [ ]:
# import DBOpt 
import MDAnalysis as mda
import numpy as np
import nglview as nv
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
import MDAnalysis.analysis.rdf
from MDAnalysis.core.topology import TransTable
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
import itertools
from itertools import *
from parmed import load_file
import pylab as pl
from scipy.spatial.distance import pdist,squareform
from Bio.SeqUtils import seq1
from Bio.SeqUtils import molecular_weight
import pytim
from skopt import gp_minimize
from sklearn.metrics import silhouette_score
# from ovito.io import import_file
# from ovito.modifiers import ConstructSurfaceModifier
# from ovito.modifiers import SelectTypeModifier

import warnings
warnings.filterwarnings('ignore')

def listtostring(s):
    str1 = ""
    for ele in s:
        str1 += ele
    return str1
def calist_listtostring(s, structure):
    str1 = ""
    for ele in s:
        for part in structure:
            if part != 'GLY':  # Replace non-'GLY' with the current residue
                str1 += ele
            else:  # Add the 'GLY' parts directly
                str1 += part
    return str1


In [ ]:
u = mda.Universe(f'/Volumes/potoyan-lab/will/hnrnpa1/75mgml/output.pdb',
                 f'/Volumes/potoyan-lab/will/hnrnpa1/75mgml/traj.dcd')

# Write the trajectory in XTC format
with mda.Writer(f'/Volumes/potoyan-lab/will/hnrnpa1/75mgml/traj.xtc', n_atoms=u.atoms.n_atoms) as W:
    for ts in u.trajectory:
        W.write(u.atoms)



In [ ]:
def dbscan_params(universe, terminal_type, sim_species, frame, chain_length, pdbu, topu):
    
    middle = math.ceil(chain_length/2)

    top_prot = topu.select_atoms('protein')
    pdb_prot = pdbu.select_atoms('protein')

    atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
    selection = " or ".join([f"(id {num} and not resname HOH)" for num in atom_list])
    chain_sel = pdbu.select_atoms(selection)

    resid_list = list(Counter(chain_sel.resids[chain_sel.resnames != 'GLY']).keys())

    metric = 'precomputed'

    param_dict = {}

    snapshot = frame.frame

    prot_selection = " or ".join([f"(protein and resnum {num} and name CA)" for num in resid_list])
    cap_prot_selection = " or ".join([f"(protein and resnum {num+1} and name CA)" for num in resid_list])
    nuc_selection = " or ".join([f"(nucleic and resnum {num} and name P)" for num in resid_list])

    if terminal_type == 'capped':
        ca_sel = universe.select_atoms(f'{prot_selection}')
    else:
        ca_sel = universe.select_atoms(f'{cap_prot_selection}')

    rna_sel = universe.select_atoms(f'{nuc_selection}')

    n_atoms = len(rna_sel) + len(ca_sel)

    tot_sel = rna_sel + ca_sel

    atoms = [atom.index for atom in rna_sel.atoms] + [atom.index for atom in ca_sel.atoms]

    to_cluster = [atom for i, atom in enumerate(universe.trajectory[snapshot].positions) if i in atoms]

    self_distances = distances.self_distance_array(tot_sel.positions, box=universe.trajectory[snapshot].dimensions)

    sq_dist_arr = np.zeros((n_atoms, n_atoms))
    triu = np.triu_indices_from(sq_dist_arr, k=1)

    sq_dist_arr[triu] = self_distances
    sq_dist_arr.T[triu] = self_distances

    # Define your objective function
    def custom_silhouette_score(params):
        param1, param2 = params
        # Your clustering function and scoring here
        model = DBSCAN(eps=param1, min_samples=param2, metric='precomputed')
        labels = model.fit_predict(sq_dist_arr)
        if len(Counter(labels)) > 1:
            score = silhouette_score(sq_dist_arr, labels)
        else:
            score = 0
        return -score

    # Define parameter space with lists of options
    param_space = [
        list(range(((2*chain_length)-1),4*chain_length)),  # Discrete parameter options
        [2,3,4,5,6,7,8,9,10,11]        # Discrete parameter options
    ]

    # Perform optimization
    result = gp_minimize(
        func=custom_silhouette_score,
        dimensions=param_space,
        n_calls=30                      # Number of function evaluations
    )

    best_eps, best_min_samples = result.x

    return best_eps, best_min_samples, metric

In [ ]:
import math

def clustering(universe, sim_species, terminal_type, frame, chain_length, pdbu, topu):

    # if chain_length < 15:
    
    middle = math.ceil(chain_length/2)

    top_prot = topu.select_atoms('protein')
    pdb_prot = pdbu.select_atoms('protein')

    atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
    selection = " or ".join([f"(id {num} and not resname HOH)" for num in atom_list])
    chain_sel = pdbu.select_atoms(selection)

    resid_list = list(Counter(chain_sel.resids[chain_sel.resnames != 'GLY']).keys())
    
    prot_selection = " or ".join([f"(protein and resnum {num} and name CA)" for num in resid_list])
    cap_prot_selection = " or ".join([f"(protein and resnum {num+1} and name CA)" for num in resid_list])
    nuc_selection = " or ".join([f"(nucleic and resnum {num} and name P)" for num in resid_list])

    # else:

    #     top_prot = topu.select_atoms('protein')
    #     pdb_prot = pdbu.select_atoms('protein')

    #     atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
    #     selection = " or ".join([f"(id {num} and not resname HOH)" for num in atom_list])
    #     chain_sel = pdbu.select_atoms(selection)

    #     resid_list = [index for index in np.arange(1,46).tolist() if ]

    if terminal_type == 'capped':
        ca_sel = universe.select_atoms(f'{prot_selection}')
    else:
        ca_sel = universe.select_atoms(f'{cap_prot_selection}')

    rna_sel = universe.select_atoms(f'{nuc_selection}')

    n_atoms = len(rna_sel) + len(ca_sel)

    tot_sel = rna_sel + ca_sel

    atoms = [atom.index for atom in rna_sel.atoms] + [atom.index for atom in ca_sel.atoms]

    to_cluster = [atom for i, atom in enumerate(universe.trajectory[frame.frame].positions) if i in atoms]

    self_distances = distances.self_distance_array(tot_sel.positions, box=frame.dimensions)

    sq_dist_arr = np.zeros((n_atoms, n_atoms))
    triu = np.triu_indices_from(sq_dist_arr, k=1)

    sq_dist_arr[triu] = self_distances
    sq_dist_arr.T[triu] = self_distances

    eps, min_num, metric = dbscan_params(universe, terminal_type, sim_species, frame, chain_length, pdbu, topu)

    model = DBSCAN(eps=eps, min_samples=min_num, metric=metric)
    
    labels = model.fit_predict(sq_dist_arr)
    b = Counter(labels)
    
    if b.most_common()[0][0] == -1:
        clustered_list = [atom for j, atom in enumerate(to_cluster)
              for i, item in enumerate(labels)
              if item == b.most_common()[1][0]
              and j == i]
    else:
        clustered_list = [atom for j, atom in enumerate(to_cluster)
              for i, item in enumerate(labels)
              if item == b.most_common()[0][0]
              and j == i]

    centroid = np.mean(clustered_list, axis=0)

    ind_list = [atoms[index] for index, j in enumerate(to_cluster)
                for i in clustered_list
                if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

    index_list = [x+1 for x in ind_list]

    res_list = [universe.atoms[i].resindex for i in index_list]

    name_list = [universe.atoms[i].resname for i in index_list]

    tot_sel_res = tot_sel.residues.resnames

    return centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels


In [ ]:
def pbc_centroid(points, box):
    """
    Compute the centroid of 3D points under periodic boundary conditions.
    
    Parameters:
    - points: numpy array of shape (N, 3), where N is the number of points.
    - box: numpy array of shape (3,), containing the box dimensions [Lx, Ly, Lz].
    
    Returns:
    - centroid: numpy array of shape (3,), the centroid under PBCs.
    """
    # Use the first point as a reference
    reference = points[0]
    
    # Unwrap points relative to the reference
    deltas = points - reference
    deltas -= np.round(deltas / box) * box  # Apply minimum image convention
    unwrapped_points = reference + deltas
    
    # Compute the centroid in unwrapped space
    centroid_unwrapped = np.mean(unwrapped_points, axis=0)
    
    # Wrap the centroid back into the primary simulation box
    centroid = centroid_unwrapped % box
    
    return centroid

In [ ]:
pdbu = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/output.pdb',
                    '/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/traj.dcd')

topu = mda.Universe(load_file('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/output.top'),
                    '/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/2_close/traj.dcd')

temp_prot = pdbu.select_atoms('protein')
chain_length = np.max(temp_prot.residues.resnums)
middle = math.ceil(chain_length/2)

top_prot = topu.select_atoms('protein')
pdb_prot = pdbu.select_atoms('protein')

atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
selection = " or ".join([f"id {num} and protein" for num in atom_list])
chain_sel = pdbu.select_atoms(selection)
structure = chain_sel.residues.resnames
n_stickers = len([item for item in structure if item != 'GLY'])

cent_dist_df = pd.DataFrame()

frame_num = 0

for frame in pdbu.trajectory[20000:20001:1]:
    
    centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(sim_species='prot', terminal_type='charged', universe=pdbu, frame=frame, chain_length=chain_length, pdbu=pdbu, topu=topu)
    
    selection = " or ".join([f"resindex {num-1} and protein" for num in res_list])
    sel_prot = pdbu.select_atoms(f'{selection}')

    clustered_list = np.array([arr.tolist() for arr in clustered_list])
    
    points = clustered_list
    box = pdbu.trajectory[frame.frame].dimensions[:3]  # Box dimensions (Lx, Ly, Lz)

    new_centroid = pbc_centroid(points, box)
    
    temp_dists = distances.distance_array(new_centroid, sel_prot.segments.atoms.positions, pdbu.trajectory[frame.frame].dimensions)
    # temp_dists = distances.distance_array(centroid, clustered_list, pdbu.dimensions)

    cent_dist_df.loc[frame.frame, 'avg_cent_dist'] = np.mean(temp_dists)

    frame_num += 1





In [ ]:
cent_dist_df

In [ ]:
import numpy as np

def pbc_centroid(points, box):
    """
    Compute the centroid of 3D points under periodic boundary conditions.
    
    Parameters:
    - points: numpy array of shape (N, 3), where N is the number of points.
    - box: numpy array of shape (3,), containing the box dimensions [Lx, Ly, Lz].
    
    Returns:
    - centroid: numpy array of shape (3,), the centroid under PBCs.
    """
    # Use the first point as a reference
    reference = points[0]
    
    # Unwrap points relative to the reference
    deltas = points - reference
    deltas -= np.round(deltas / box) * box  # Apply minimum image convention
    unwrapped_points = reference + deltas
    
    # Compute the centroid in unwrapped space
    centroid_unwrapped = np.mean(unwrapped_points, axis=0)
    
    # Wrap the centroid back into the primary simulation box
    centroid = centroid_unwrapped % box
    
    return centroid

# Example usage
points = clustered_list
box = pdbu.trajectory[20000].dimensions[:3]  # Box dimensions (Lx, Ly, Lz)

centroid = pbc_centroid(points, box)
print("Centroid with PBCs:", centroid)


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pdbu.trajectory[20000]

# Separate the list into x, y, z coordinates
x, y, z = zip(*sel_prot.segments.atoms.positions)

# Create a 3D plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of the points
ax.scatter(x, y, z, c='blue', marker='o', alpha=0.015)

ax.scatter(new_centroid[0], new_centroid[1], new_centroid[2], c='red', marker='o', s=50)

# Label the axes
ax.set_xlabel('X Axis')
ax.set_ylabel('Y Axis')
ax.set_zlabel('Z Axis')

In [ ]:
def cluster_check(u, start=17154, stop=17155, step=1, sim_species='prot', terminal_type='charged'):

    ###  This function will (1)find the largest cluster for each    ###
    ###  frame in a trajectory, (2)make a universe containing only  ###
    ###  the residues inside that cluster (pdb file), (3)visualize  ###
    ###  the cluster in red with the rest of the residues in purple ###
    
    # start = frame to start the analysis on (should be after equilibration portion of simulation
    # stop = frame to stop the analysis on
    # step = stride to use to get from start to stop
    
    temp_u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/output.pdb')
    temp_prot = u.select_atoms('protein')
    chain_length = np.max(temp_prot.residues.resnums)
    middle = math.ceil(chain_length/2)
    
    for frame in u.trajectory[start:stop:step]:

        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(u, sim_species, terminal_type, frame, chain_length)

#         if len(clustered_list) > 35:

        columns = ['x','y','z']
        all_df = pd.DataFrame(to_cluster, columns=columns)
        clustered_df = pd.DataFrame(clustered_list, columns=columns)

        import time
        import pylab as pl
        from IPython import display
        from ipywidgets import interact
        display.clear_output(wait=True)
        fig = px.scatter_3d(all_df, x='x', y='y', z='z')
        fig.add_trace(px.scatter_3d(clustered_df, x='x', y='y', z='z', color_discrete_sequence=['red']).data[0])
        fig.show();
        display.display(plt.gcf())
        time.sleep(0.5)

#         else:
#             continue

In [ ]:
def cluster_rdf(universe, sim_species, terminal_type, start, stop, step, output):

    frame_num = 0
    for frame in universe.trajectory[start:stop:step]:
        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(universe, sim_species, terminal_type, frame)

        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]
        sel = universe.select_atoms(selection)
        
        with mda.Writer(f'cluster.pdb', bonds='all') as pdb:
            pdb.write(sel)

        univ = mda.Universe(f'cluster.pdb', guess_bonds=True)

        centroid = np.mean(univ.atoms.positions, axis=0)
        
        dists = distances.distance_array(centroid, univ.atoms.positions, univ.dimensions)

        index = np.where(dists == dists.min())

        pos = univ.atoms.positions[index[1][0]]

        res_list = ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
                   'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']
        
        df = pd.DataFrame()
        
        for i in ['ARG']:
            u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/equal/output.gro')
            prot = u.select_atoms('protein')
            chain_length = np.max(prot.residues.resnums)
            middle = math.ceil(chain_length/2)
            universe.trajectory[frame.frame]
            a_group = univ.select_atoms(f'prop x == {pos[0]} and prop y == {pos[1]} and prop z == {pos[2]}')
            b_group = univ.select_atoms(f'resname {i} and resnum {middle} and name CA and not segid {a_group.segids.tolist()[0]}')
            rdf = mda.analysis.rdf.InterRDF(a_group, b_group, nbins=25, range=[1,25], exclude_same='segment')
            rdf.run()
            if not 'Radius' in df:
                df['Radius'] = rdf.bins
            df[i] = rdf.rdf

        array = df.to_numpy()
        columns = list(df.columns.values)

        if frame_num == 0:
            final_array = array
            frame_num += 1
        else:
            final_array = np.dstack((final_array, array))
            frame_num += 1

    avg_array = np.nanmean(final_array, axis=2)

    cluster_rdf_df = pd.DataFrame(avg_array, columns=columns)
#     cluster_rdf_df = pd.DataFrame(final_array, columns=columns)

#     cluster_rdf_df.to_csv(output)
    return cluster_rdf_df


In [ ]:
def phase_diagram(a, sim_species, terminal_type, start, stop, step, dens=125):
        
    phases_df = pd.DataFrame(columns=['dense_phase_density_mgml','dilute_phase_density_mgml'])
    
    for frame in u.trajectory[start:stop:step]:
        
        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(sim_species, terminal_type, frame, dens)
        
        selection = 'resindex '
        for item in res_list:
            selection += str(item)+ ' or resindex '
        selection = selection[0:-13]
        sel = u.select_atoms(selection)

        with mda.Writer('./cluster.pdb') as pdb:
            pdb.write(sel)

        univ = mda.Universe('./cluster.pdb', guess_bonds=True)
        
        box_volume_ml = (univ.dimensions[0]*univ.dimensions[1]*univ.dimensions[2])/ (1.0E24)
        
        prot = u.select_atoms('protein')
        system_list = prot.residues.resnames.tolist()
        box_str = listtostring(system_list)
        new_box_str = seq1(box_str)
        box_mass_amu = molecular_weight(new_box_str,'protein')
        
        box_mass_mg = box_mass_amu / 6.023e20

        pipeline = import_file('./cluster.pdb')
        pipeline.modifiers.append(ConstructSurfaceModifier(radius = 10, smoothing_level = 10, 
            only_selected = False, identify_regions = True, map_particles_to_regions = True))
        data = pipeline.compute()

        cluster_volume_ml = (data.attributes['ConstructSurfaceMesh.filled_volume']) / (1.0E24)
        
#         cluster_list = univ.residues.resnames.tolist()
        cluster_list = name_list
        cluster_str = listtostring(cluster_list)
        new_cluster_str = seq1(cluster_str)
        cluster_mass_amu = molecular_weight(new_cluster_str,'protein')
        
        cluster_mass_mg = cluster_mass_amu / 6.023e20
        
        dilute_mass_mg = box_mass_mg - cluster_mass_mg
        
        dilute_volume_ml = box_volume_ml - cluster_volume_ml
        
        cluster_dens = cluster_mass_mg / cluster_volume_ml
        dilute_dens = dilute_mass_mg / dilute_volume_ml
        
        phases_df.loc[frame.frame, 'dense_phase_density_mgml'] = cluster_dens
        phases_df.loc[frame.frame, 'dilute_phase_density_mgml'] = dilute_dens

    return phases_df

In [ ]:
import math

# UNIVERSE NEEDS TO BE MADE WITH .PDB FILE, NOT .GRO! #

def layer_comp(sim_species, terminal_type, u, system, layers, start, stop, step):
    
    polar = ['SER','THR','CYS','ASN','GLN']
    nonpolar = ['GLY','ALA','PRO','VAL','LEU','ILE','MET']
    aromatic = ['PHE','TRP','TYR']
    charged = ['HIS','LYS','ARG','ASP','GLU']

    if sim_species == 'prot+rna':
        core_dict = {'ALA': 0, 'ARG': 0, 'ASP': 0, 'ASN': 0, 'HIS': 0, 'LYS': 0, 'PRO': 0, 'GLY': 0, 
                    'GLU': 0, 'THR': 0, 'SER': 0, 'GLN': 0, 'CYS': 0, 'TYR': 0, 'TRP': 0, 'PHE': 0, 
                    'VAL': 0, 'MET': 0, 'ILE': 0, 'LEU': 0, 'U': 0, 'G': 0, 'T': 0, 'C': 0, 'A': 0}
    elif sim_species == 'prot':
        core_dict = {'ALA': 0, 'ARG': 0, 'ASP': 0, 'ASN': 0, 'HIS': 0, 'LYS': 0, 'PRO': 0, 'GLY': 0,
                    'GLU': 0, 'THR': 0, 'SER': 0, 'GLN': 0, 'CYS': 0, 'TYR': 0, 'TRP': 0, 'PHE': 0,
                    'VAL': 0, 'MET': 0, 'ILE': 0, 'LEU': 0}
    else:
        core_dict = {'U': 0, 'G': 0, 'T': 0, 'C': 0, 'A': 0}

    middle_dict = core_dict.copy()
    shell_dict = core_dict.copy()

    core_charged_res = []
    middle_charged_res = []
    shell_charged_res = []

    dist_list_core = []
    dist_list_middle = []
    dist_list_shell = []

    hphob_dict = {'ILE':0.972973,'VAL':0.891892,'LEU':0.972973,'PHE':1.0,'CYS':0.594595,
                  'MET':0.837838,'ALA':0.72973,'GLY':0.648649,'THR':0.675676,
                  'SER':0.594595,'TRP':0.945946,'TYR':0.864865,'PRO':1.0,'HIS':0.513514,
                  'GLU':0.459459,'GLN':0.513514,'ASP':0.378378,'ASN':0.432432,
                  'LYS':0.513514,'ARG':0.0}
    
    frames = 1
    
    temp_u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/{system}/output.pdb')
    temp_prot = u.select_atoms('protein')
    chain_length = np.max(temp_prot.residues.resnums)
    middle = math.ceil(chain_length/2)    
   
    if chain_length < 9:
        alpha = 12
    elif chain_length >= 9 and chain_length < 12:
        alpha = 15
    elif chain_length >= 12 and chain_length < 18:
        alpha = 18
    elif chain_length >= 18 and chain_length < 24:
        alpha = 24
    
    for frame in u.trajectory[start:stop:step]:
        
        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(u, sim_species, terminal_type, frame, chain_length)
        
        selection = 'resindex '
        for item in res_list:
            selection += str(item) + ' or resindex '
        selection = selection[0:-13]
        sel = u.select_atoms(selection)

        with mda.Writer('/Users/williambrown/Desktop/spatial.pdb') as pdb:
            pdb.write(sel)

        univ = mda.Universe('/Users/williambrown/Desktop/spatial.pdb')
        
        group = univ.select_atoms(f'(resnum {middle} and name CA) or (resnum {middle} and name P)')
    
        interface = pytim.GITIM(univ, group=group, alpha=alpha, max_layers=layers, molecular=False, centered='middle')
        
        if len(interface.layers[2].residues) > 5:
            
            for i,j in Counter(interface.layers[2].resnames).items():
                for k,l in core_dict.items():
                    if i == k:
                        core_dict[k] += j
            for i,j in Counter(interface.layers[1].resnames).items():
                for k,l in middle_dict.items():
                    if i == k:
                        middle_dict[k] += j
            for i,j in Counter(interface.layers[0].resnames).items():
                for k,l in shell_dict.items():
                    if i == k:
                        shell_dict[k] += j
            core_charge = 0
            for i,j in Counter(interface.layers[2].resnames).items():
                if i in charged:
                    core_charge += j
                elif i in polar:
                    core_charge += j
            core_charged_res.append(core_charge/(len(interface.layers[2].resnames)))

            middle_charge = 0
            for i,j in Counter(interface.layers[1].resnames).items():
                if i in charged:
                    middle_charge += j
                elif i in polar:
                    middle_charge += j
            middle_charged_res.append(middle_charge/(len(interface.layers[1].resnames)))

            shell_charge = 0
            for i,j in Counter(interface.layers[0].resnames).items():
                if i in charged:
                    shell_charge += j
                elif i in polar:
                    shell_charge += j
            shell_charged_res.append(shell_charge/(len(interface.layers[0].resnames)))

            centroid = np.mean(univ.atoms.positions, axis=0)

            dist_list_core.append(np.mean([np.sqrt(((i[0]-centroid[0])**2) + 
                                                   ((i[1]-centroid[1])**2) + 
                                                   ((i[2]-centroid[2])**2)) 
                                                    for i in interface.layers[2].positions])/10)

            dist_list_middle.append(np.mean([np.sqrt(((i[0]-centroid[0])**2) + 
                                                   ((i[1]-centroid[1])**2) + 
                                                   ((i[2]-centroid[2])**2)) 
                                                    for i in interface.layers[1].positions])/10)

            dist_list_shell.append(np.mean([np.sqrt(((i[0]-centroid[0])**2) + 
                                                   ((i[1]-centroid[1])**2) + 
                                                   ((i[2]-centroid[2])**2)) 
                                                    for i in interface.layers[0].positions])/10)

            frames += 1
    
    for layer in [core_dict, middle_dict, shell_dict]:
        for i,j in layer.items():
            layer[i] /= frames
    
    core_df = pd.DataFrame.from_dict(core_dict, orient='index', columns=['Avg_Count'])
    core_df.index.name = 'Residue'
    core_df = core_df.sort_values(by='Avg_Count')
    for i,j in hphob_dict.items():
        core_df.loc[i,'Hydrophobicity'] = j
    core_df['lambda'] = core_df['Avg_Count'] * core_df['Hydrophobicity']
    
    middle_df = pd.DataFrame.from_dict(middle_dict, orient='index', columns=['Avg_Count'])
    middle_df.index.name = 'Residue'
    middle_df = middle_df.sort_values(by='Avg_Count')
    for i,j in hphob_dict.items():
        middle_df.loc[i,'Hydrophobicity'] = j
    middle_df['lambda'] = middle_df['Avg_Count'] * middle_df['Hydrophobicity']
    
    shell_df = pd.DataFrame.from_dict(shell_dict, orient='index', columns=['Avg_Count'])
    shell_df.index.name = 'Residue'
    shell_df = shell_df.sort_values(by='Avg_Count')
    for i,j in hphob_dict.items():
        shell_df.loc[i,'Hydrophobicity'] = j
    shell_df['lambda'] = shell_df['Avg_Count'] * shell_df['Hydrophobicity']
    
    core_value = core_df['lambda'].sum() / sum(core_dict.values())

    middle_value = middle_df['lambda'].sum() / sum(middle_dict.values())

    shell_value = shell_df['lambda'].sum() / sum(shell_dict.values())
    
    core_dist_value = "%.1f" % (np.mean(dist_list_core))
    
    mid_dist_value = "%.1f" % (np.mean(dist_list_middle))
    
    shell_dist_value = "%.1f" % (np.mean(dist_list_shell))
    
    layer_dict = {core_dist_value:core_value, 
                  mid_dist_value:middle_value, 
                  shell_dist_value:shell_value}
    
    charge_dict = {core_dist_value:np.mean(core_charged_res), 
                   mid_dist_value:np.mean(middle_charged_res), 
                   shell_dist_value:np.mean(shell_charged_res)}
    
    return core_df, middle_df, shell_df, layer_dict, charge_dict, interface, univ, frames



In [ ]:
# total_df = pd.DataFrame(index=['core', 'intermediate', 'surface'])

for system in ['capped']:

    u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/og_gxg/{system}/125_capped_output.pdb',
                    f'/Volumes/potoyan-lab/will/gxg-project/og_gxg/{system}/125_capped.dcd')

    core_df, middle_df, shell_df, layer_dict, charge_dict, interface, univ, frames = layer_comp(
        sim_species='prot', terminal_type='charged', system=system, u=u, layers=3, 
        start=4000, stop=-1, step=250)

    total_df.loc['core',f'{system}_hydrophobic'] = list(layer_dict.values())[0]
    total_df.loc['intermediate',f'{system}_hydrophobic'] = list(layer_dict.values())[1]
    total_df.loc['surface',f'{system}_hydrophobic'] = list(layer_dict.values())[2]

    total_df.loc['core',f'{system}_charge'] = list(charge_dict.values())[0]
    total_df.loc['intermediate',f'{system}_charge'] = list(charge_dict.values())[1]
    total_df.loc['surface',f'{system}_charge'] = list(charge_dict.values())[2]

    fig, ax1 = plt.subplots(figsize=[6.5,5])

    ax1.plot(list(layer_dict.keys())[0], list(layer_dict.values())[0], marker='*', linewidth=3, markersize=25, linestyle='-', color='#d03535')
    ax1.plot(list(layer_dict.keys())[1], list(layer_dict.values())[1], marker='*', linewidth=3, markersize=25, linestyle='-', color='#3671a5')
    ax1.plot(list(layer_dict.keys())[2], list(layer_dict.values())[2], marker='*', linewidth=3, markersize=25, linestyle='-', color='darkorchid')

    ax1.plot(layer_dict.keys(), layer_dict.values(), marker='*', linewidth=3, markersize=0, linestyle=':', color='gray', alpha=0.65)

    ax1.plot(list(charge_dict.keys())[0], list(charge_dict.values())[0], marker='o', linewidth=3, markersize=21, linestyle='-', color='#d03535')
    ax1.plot(list(charge_dict.keys())[1], list(charge_dict.values())[1], marker='o', linewidth=3, markersize=21, linestyle='-', color='#3671a5')
    ax1.plot(list(charge_dict.keys())[2], list(charge_dict.values())[2], marker='o', linewidth=3, markersize=21, linestyle='-', color='darkorchid')

    ax1.plot(charge_dict.keys(), charge_dict.values(), marker='*', linewidth=3, markersize=0.000001, linestyle=':', color='gray', alpha=0.65)

    # ax1.set_ylim(0.41,0.74)
    ax1.set_xlabel('Distance from Cluster Centroid (nm)', fontsize=14)
    ax1.set_ylabel('Avg Residue Hydrophobicity', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    ax2 = ax1.twinx()

    # ax2.set_ylim(0.41,0.74)
    ax2.set_ylabel('Charged & Polar Residue Fraction', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    list_color  = ["black", "black"]
    list_mak    = ["o","*"]
    list_lab    = ["Charged/Polar Residue Fraction", "Avg Residue Hydrophobicity"]


    from matplotlib.legend_handler import HandlerBase

    class MarkerHandler(HandlerBase):
        def create_artists(self, legend, tup, xdescent, ydescent,
                            width, height, fontsize, trans):
            return [plt.Line2D([width/2], [height/2.],ls="",
                           marker=tup[1],color=tup[0], transform=trans)]

    legend = plt.legend(list(zip(list_color,list_mak)), list_lab, fontsize=13,
              bbox_to_anchor=(0.725, 1.01),
              handler_map={tuple:MarkerHandler()},
              markerscale=20)


#     legend.legendHandles[0].set_markersize(12)
#     legend.legendHandles[1].set_markersize(17)

    # plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/9res_{system}_spatial_org.png', dpi=500, transparent=True)
    # plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/9res_{system}_spatial_org.pdf', dpi=500, transparent=True)

total_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/capped_all_systems_hphob_charge.csv')


In [ ]:
capped = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/capped_all_systems_hphob_charge.csv', index_col=0)

res3 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/3res_all_systems_hphob_charge.csv', index_col=0)

res9 = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/9res_all_systems_hphob_charge.csv', index_col=0)

layer_dict_3res = {'equal':['1.6','2.2','2.9'], 'charged':['1.6','2.2','2.9'], 
              'aromatic':['1.6','2.3','3.0'], 'polar':['1.5','2.1','2.8'],
              'nonpolar':['1.3','2.0','2.8']}

layer_dict_9res = {'equal':['1.4','2.2','2.9'], 'charged':['1.0','1.7','2.4'], 
              'aromatic':['0.9','1.6','2.3'], 'polar':['1.1','1.7','2.4'],
              'nonpolar':['1.0','1.6','2.3']}

avg_layer_list = ['8-16','16-23','23-31']



In [ ]:
fig, ax = plt.subplots(figsize=[7,5])

colors = ['gray','#b30039','#e3ba14','#7aaa3d','#ac5cad']

system_list = ['equal','charged','aromatic','polar','nonpolar']

labels = [i.capitalize() for i in system_list]

for i,system in enumerate(system_list):
    ax.plot(avg_layer_list, res3[f'{system}_charge']*100, color=colors[i], marker='o', linewidth=3, markersize=12, linestyle=':', alpha=0.8)

plt.ylim(13,92)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.set_xlabel("Distance from Cluster Centroid (Å)", fontsize=14)
ax.set_ylabel("% Clustered Charged/Polar Residues", fontsize=14)

patches = [plt.Rectangle((0, 0), 1, 1, color=color, alpha=0.7) for color in colors]
ax.legend(patches, labels, fontsize=11.8, framealpha=0.5, loc='upper right', ncols=3)

fig.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/pdfs/all_3res.pdf', dpi=500, transparent=True)
fig.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/pngs/all_3res.png', dpi=500, transparent=True)

In [ ]:
for system in ['equal','charged','aromatic','polar','nonpolar']:

    fig, ax1 = plt.subplots(figsize=[6.5,5])

    ax1.plot(layer_dict[f'{system}'][0], tot_df[f'{system}_hydrophobic'][0], marker='*', linewidth=3, markersize=25, linestyle='-', color='#d03535')
    ax1.plot(layer_dict[f'{system}'][1], tot_df[f'{system}_hydrophobic'][1], marker='*', linewidth=3, markersize=25, linestyle='-', color='#3671a5')
    ax1.plot(layer_dict[f'{system}'][2], tot_df[f'{system}_hydrophobic'][2], marker='*', linewidth=3, markersize=25, linestyle='-', color='darkorchid')

    ax1.plot(layer_dict[f'{system}'], tot_df[f'{system}_hydrophobic'], marker='*', linewidth=3, markersize=0, linestyle=':', color='gray', alpha=0.65)

    ax1.plot(layer_dict[f'{system}'][0], tot_df[f'{system}_charge'][0], marker='o', linewidth=3, markersize=21, linestyle='-', color='#d03535')
    ax1.plot(layer_dict[f'{system}'][1], tot_df[f'{system}_charge'][1], marker='o', linewidth=3, markersize=21, linestyle='-', color='#3671a5')
    ax1.plot(layer_dict[f'{system}'][2], tot_df[f'{system}_charge'][2], marker='o', linewidth=3, markersize=21, linestyle='-', color='darkorchid')

    ax1.plot(layer_dict[f'{system}'], tot_df[f'{system}_charge'], marker='*', linewidth=3, markersize=0.000001, linestyle=':', color='gray', alpha=0.65)

    # ax1.set_ylim(0.41,0.74)
    ax1.set_xlabel('Distance from Cluster Centroid (nm)', fontsize=14)
    ax1.set_ylabel('Avg Residue Hydrophobicity', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    ax2 = ax1.twinx()

    # ax2.set_ylim(0.41,0.74)
    ax2.set_ylabel('Charged & Polar Residue Fraction', fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    list_color  = ["black", "black"]
    list_mak    = ["o","*"]
    list_lab    = ["Charged/Polar Residue Fraction", "Avg Residue Hydrophobicity"]

    if system == 'equal':
        loc = 'upper left'
    elif system == 'charged':
        loc = 'center left'
    elif system == 'aromatic':
        loc = 'center'
    elif system == 'polar':
        loc = 'center left'
    else:
        loc = 'center'
    
    from matplotlib.legend_handler import HandlerBase

    class MarkerHandler(HandlerBase):
        def create_artists(self, legend, tup, xdescent, ydescent,
                            width, height, fontsize, trans):
            return [plt.Line2D([width/2], [height/2.],ls="",
                           marker=tup[1],color=tup[0], transform=trans)]

    legend = plt.legend(list(zip(list_color,list_mak)), list_lab, fontsize=13,
              loc=loc,
              handler_map={tuple:MarkerHandler()})

    plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/pngs/3res_{system}_spatial_org.png', transparent=True, dpi=500)
    plt.savefig(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/charge_hydrophobicity/pdfs/3res_{system}_spatial_org.pdf', transparent=True, dpi=500)

#     legend.legend_handles[0].set_markersize = 25
#     legend.legend_handles[1].set_markersize = 25
    

In [ ]:
ca_sel = univ.select_atoms('name CA')

coord = 'x'

if coord == 'x':
    coord_num = 0
elif coord == 'y':
    coord_num = 1
else:
    coord_num = 2

sliced_sel = univ.select_atoms(f'name CA and prop {coord} < ' + str(round(ca_sel.centroid()[coord_num], 1)-4))

condensate_slice = univ.select_atoms(f'name CA and prop {coord} < ' + str(round(ca_sel.centroid()[coord_num], 1)-4)).atoms.indices
to_color = univ.residues[interface.layers[2].resindices].atoms.indices
new_indices_to_color = np.where(np.in1d(condensate_slice,to_color))[0]
sel=",".join([str(i) for i in new_indices_to_color])
core_selection=f'@{sel}'

condensate_slice = univ.select_atoms(f'name CA and prop {coord} < ' + str(round(ca_sel.centroid()[coord_num], 1)-4)).atoms.indices
to_color = univ.residues[interface.layers[1].resindices].atoms.indices
new_indices_to_color = np.where(np.in1d(condensate_slice,to_color))[0]
sel=",".join([str(i) for i in new_indices_to_color])
middle_selection=f'@{sel}'

condensate_slice = univ.select_atoms(f'name CA and prop {coord} < ' + str(round(ca_sel.centroid()[coord_num], 1)-4)).atoms.indices
to_color = univ.residues[interface.layers[0].resindices].atoms.indices
new_indices_to_color = np.where(np.in1d(condensate_slice,to_color))[0]
sel=",".join([str(i) for i in new_indices_to_color])
shell_selection=f'@{sel}'

view = nv.show_mdanalysis(sliced_sel)
view.add_spacefill(radius=0.001)
view.add_spacefill(core_selection, color='firebrick', opacity=0.9, radius=4)
view.add_spacefill(middle_selection, color='steelblue', opacity=.7, radius=4)
view.add_spacefill(shell_selection, color='purple', opacity=.5, radius=4)
view

In [ ]:
import time
def generate_images(v=view):
    v.clear()
    v.add_cartoon(color='red')
    im0 = v.render_image()
    v.clear()
    v.add_cartoon(color='blue')
    im1 = v.render_image()
    for im in [im0, im1]:
        while not im.value:
            time.sleep(0.1)
    for n, im in zip('ab', [im0, im1]):
        with open(f'figure_{n}.png', 'wb') as fh:
            fh.write(im.value)

import threading
thread = threading.Thread(
    target=generate_images,
)
thread.daemon = True
thread.start()


In [ ]:
name_list

In [ ]:
def cluster_water(u, sim_species, terminal_type, start, stop, step, pdbu, topu):

    percentage_df = pd.DataFrame()
    mass_ratio_list = []
    
    temp_prot = u.select_atoms('protein')
    chain_length = np.max(temp_prot.residues.resnums)
    middle = math.ceil(chain_length/2)
    
    top_prot = topu.select_atoms('protein')
    pdb_prot = pdbu.select_atoms('protein')
    
    atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
    selection = " or ".join([f"(id {num} and not resname HOH)" for num in atom_list])
    chain_sel = pdbu.select_atoms(selection)
    structure = chain_sel.residues.resnames

    for frame in u.trajectory[start:stop:step]:

        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(u, terminal_type, sim_species, frame, chain_length, pdbu, topu)

        if len(clustered_list) > 25:

            aa_str = calist_listtostring(name_list, structure)

            new_aa_str = seq1(aa_str)

            pep_mass_amu = molecular_weight(new_aa_str,'protein')

            selection = " or ".join([f"(resindex {num})" for num in res_list])

            sel_prot = u.select_atoms(f'({selection})' + ' or resname HOH')

            with mda.Writer("/Volumes/potoyan-lab/will/gxg-project/notebooks/cluster.pdb", sel_prot.segments.atoms) as pdb:
                pdb.write(sel_prot.segments.atoms)

            temp_univ = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/notebooks/cluster.pdb')

            total_sel = temp_univ.select_atoms('protein or (resname HOH and around 5 protein)')

            with mda.Writer("/Volumes/potoyan-lab/will/gxg-project/notebooks/cluster.pdb") as pdb:
                pdb.write(total_sel)

            univ = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/notebooks/cluster.pdb')

            water = univ.select_atoms('resname HOH')

            n_water = water.n_residues

            mass_ratio = ((n_water*18.015) / pep_mass_amu)

            mass_ratio_list.append(mass_ratio)

        else:
            continue

    percentage_df[f'Mass Ratio'] = mass_ratio_list

    return percentage_df, univ, temp_univ

In [ ]:
thing = '9_res/equal/'

if 'close' not in thing and 'spread' not in thing:
    print('regular')
else:
    print('different')

In [ ]:
pdbu = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/charged/output.pdb',
        f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/charged/trajectory.dcd')

topu = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/charged/output.top'),
        f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_res/charged/trajectory.dcd')

In [ ]:
def cluster_asphericity(system, sim_species, terminal_type, start, stop, step, pdbu, topu):

    if '3_res' in system:

        pdbu = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/trajectory.dcd')
    
        topu = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.top'),
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/trajectory.dcd')

    else:

        pdbu = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/traj.dcd')
    
        topu = mda.Universe(load_file(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.top'),
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/traj.dcd')

    asphericity_list = []
    
    temp_prot = pdbu.select_atoms('protein')
    chain_length = np.max(temp_prot.residues.resnums)
    middle = math.ceil(chain_length/2)
    
    top_prot = topu.select_atoms('protein')
    pdb_prot = pdbu.select_atoms('protein')
    
    atom_list = pdb_prot.atoms.ids[top_prot.atoms.fragindices == int(np.max(top_prot.atoms.fragindices)/2)]
    selection = " or ".join([f"(id {num} and not resname HOH)" for num in atom_list])
    chain_sel = pdbu.select_atoms(selection)
    structure = chain_sel.residues.resnames

    for frame in pdbu.trajectory[start:stop:step]:

        centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(pdbu, terminal_type, sim_species, frame, chain_length, pdbu, topu)

        selection = " or ".join([f"(resindex {num})" for num in res_list])
        
        sel_prot = pdbu.select_atoms(f'({selection})')

        with mda.Writer("/Users/williambrown/Desktop/cluster.pdb", sel_prot.segments.atoms) as pdb:
            pdb.write(sel_prot.segments.atoms)

        temp_univ = mda.Universe('/Users/williambrown/Desktop/cluster.pdb')

        asphericity_sel = temp_univ.select_atoms('protein')
        # asphericity_sel = temp_univ.select_atoms('protein and name CA')

        asphericity = asphericity_sel.asphericity(wrap=True)

        asphericity_list.append(asphericity)

    average_asphericity = np.nanmean(asphericity_list)
    
    return average_asphericity, asphericity_list
        


In [ ]:
import os
os.environ["MKL_VERBOSE"] = "0"

In [ ]:
systems = ['3_res/equal','3_res/charged','3_res/aromatic','3_res/polar','3_res/nonpolar',
          '9_res/equal','9_res/charged','9_res/aromatic','9_res/polar','9_res/nonpolar',
          '9_res/equal/2_close','9_res/equal/2_spread','9_res/equal/3_close','9_res/equal/3_spread']

asphericity_time_df = pd.DataFrame()
avg_asphericity_df = pd.DataFrame()

for system in systems:

    print(system)

    if '3_res' in system:

        frames_u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/trajectory.dcd')
    else:

        frames_u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/output.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{system}/traj.dcd')

    average_asphericity, asphericity_list = cluster_asphericity(system=system, sim_species='prot', terminal_type='charged', start=4000, stop=-1, step=int(frames_u.trajectory.n_frames/1000), pdbu=pdbu, topu=topu)
    
    if 'close' in system:
        
        new_column = pd.DataFrame({f'X{system[12]}': asphericity_list})

        avg_asphericity_df.loc['G(r)',f'X{system[12]}'] = average_asphericity
        
        asphericity_time_df = pd.concat([asphericity_time_df, new_column], axis=1)
        
    elif 'spread' in system:

        new_column = pd.DataFrame({f'X{system[12]}S': asphericity_list})

        avg_asphericity_df.loc['G(r)',f'X{system[12]}S'] = average_asphericity
        
        asphericity_time_df = pd.concat([asphericity_time_df, new_column], axis=1)

    else:
        new_column = pd.DataFrame({f'{system[0]}{system[6].upper()}': asphericity_list})
        
        avg_asphericity_df.loc['G(r)',f'{system[0]}{system[6].upper()}'] = average_asphericity
        
        if len(asphericity_time_df.columns) == 0:
            asphericity_time_df[f'{system[0]}{system[6].upper()}'] = asphericity_list
        else:
            asphericity_time_df = pd.concat([asphericity_time_df, new_column], axis=1)

    print(asphericity_time_df)



In [ ]:
avg_asphericity_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/asphericity_data.csv')
asphericity_time_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/asphericity_time_data.csv')

In [ ]:
# prot = pdbu.select_atoms('protein')
view = nv.show_mdanalysis(rg_sel)
view.add_ball_and_stick()
view.add_unitcell()
view

In [ ]:
topu.trajectory[30802]
prot = topu.select_atoms('protein')
with mda.Writer('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/3_close_full_snap.pdb', n_atoms=prot.atoms.n_atoms) as pdb:
    pdb.write(prot)

In [ ]:
tot_water_df = pd.DataFrame()

for length in ['3_res','9_res']:

    if length == '3_res':

        name = '3res'
    
        for system in ['equal','aromatic']:
        
            # if system == 'aromatic':
            #     u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
            #                     f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/traj.dcd',
            #                     f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/traj_2.dcd')
            # else:
            #     u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
            #                     f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/trajectory.dcd')

            
            # temp_df = cluster_water(u, sim_species='prot', terminal_type='charged', start=4000, stop=-1, step=1000)
            
            tot_water_df[f'{name}_{system} Mass Ratio'] = temp_df[f'Mass Ratio']
    
            tot_water_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/new_total_water_df.csv')

    else:
    
        for system in ['charged','polar']:
        
            name = '9res'
            u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/traj.dcd')
            
            temp_df = cluster_water(u, sim_species='prot', terminal_type='charged', start=4000, stop=-1, step=1000)
            
            tot_water_df[f'{name}_{system} Mass Ratio'] = temp_df[f'Mass Ratio']
    
            tot_water_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/new_total_water_df.csv')


In [ ]:
tot_water_df

In [ ]:
tot_water_df.to_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/new_total_water_df.csv')

In [ ]:
tot_water_df = pd.read_csv('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/new_total_water_df.csv', index_col=0)

In [ ]:
tot_water_df

In [ ]:
tot_water_df = tot_water_df.dropna()

In [ ]:
tot_water_df.columns

In [ ]:
# tot_water_df = tot_water_df.rename(columns={'3res_equal Mass Ratio':'3E', 
#                                          '3res_charged Mass Ratio':'3C',
#                                          '3res_aromatic Mass Ratio':'3A', 
#                                          '3res_polar Mass Ratio':'3P',
#                                          '3res_nonpolar Mass Ratio':'3N', 
#                                          '9res_equal Mass Ratio':'9E',
#                                          '9res_charged Mass Ratio':'9C', 
#                                          '9res_aromatic Mass Ratio':'9A',
#                                          '9res_polar Mass Ratio':'9P', 
#                                          '9res_nonpolar Mass Ratio':'9N'})

tot_water_df = tot_water_df.rename(columns={'9res_ Mass Ratio':'GGGGXGGGG', '9res_2_close/ Mass Ratio':'GGGXGXGGG',
       '9res_2_spread/ Mass Ratio':'GXGGGGGXG', '9res_3_close/ Mass Ratio':'GGGXXXGGG',
       '9res_3_spread/ Mass Ratio':'GXGGXGGXG'})
                                            

In [ ]:
ax = plt.subplots(figsize=[7,5])
tot_water_df.boxplot(grid=False)
plt.xticks(rotation=25, fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('System', fontsize=14)
plt.ylabel('Water/Peptide Mass Ratio', fontsize=14)
plt.tight_layout()
plt.ylim(0.749,2.8)

plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/res_placement_water_peptide_cluster_ratio.png', dpi=500, transparent=True)
plt.savefig('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/figures/water/res_placement_water_peptide_cluster_ratio.pdf', dpi=500, transparent=True)


plt.show();


In [ ]:
u = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/equal/output.pdb')

prot = u.select_atoms('resname ARG')

segs = prot.segments


In [ ]:
with mda.Writer('/Users/williambrown/Desktop/final_figures/pymol_figs/grg_segments.pdb') as pdb:
    pdb.write(segs)

In [ ]:
import math

u = mda.Universe('/Volumes/potoyan-lab/will/gxg-project/og_gxg/50mgml_100a/replica-1/output.gro',
                '/Volumes/potoyan-lab/will/gxg-project/og_gxg/50mgml_100a/replica-1/4ms-traj.dcd')

prot = u.select_atoms('protein')
chain_length = np.max(prot.residues.resids)
middle = math.ceil(chain_length/2)

centroid, to_cluster, clustered_list, res_list, name_list, eps, min_num, labels = clustering(universe=u, terminal_type='charged', sim_species='prot', frame=u.trajectory[94949], chain_length=chain_length)

selection = 'resindex '
for item in res_list:
    selection += str(item) + ' or resindex '
selection = selection[0:-13]
sel_prot = u.select_atoms(selection + ' or resname HOH')

with mda.Writer(f'cluster.pdb') as pdb:
    pdb.write(sel_prot)

temp_univ = mda.Universe('cluster.pdb')

total_sel = temp_univ.select_atoms('protein or (resname HOH and around 5 protein)')

with mda.Writer('/Users/williambrown/Desktop/final_figures/pymol_figs/50_water_cluster.pdb') as pdb:
    pdb.write(total_sel)





